# SDR Modulation Classifier — Exploratory Notebook

This notebook walks through:
1. Generating synthetic IQ data
2. Feature extraction and visualisation
3. Training and evaluating the Random Forest model
4. Confusion matrix and feature importance plots

In [ ]:
import sys
sys.path.insert(0, '../../backend')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

from app.services.sdr_service import generate_demo_iq, DEMO_MODULATIONS, _GENERATORS
from app.services.ml_service import extract_features
from scripts.train_model import generate_single_modulation_iq, SAMPLES_PER_CLASS, NUM_IQ_SAMPLES

FEATURE_NAMES = [
    'spectral_entropy', 'spectral_flatness', 'peak_to_mean',
    'kurt_i', 'kurt_q', 'skew_i', 'skew_q', 'iq_imbalance',
    'am_std', 'fm_std', 'pm_std', 'envelope_mean'
]
print('✅ Imports OK')

In [ ]:
# ── Plot example IQ waveforms for each modulation ─────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for idx, mod in enumerate(DEMO_MODULATIONS):
    i_s, q_s = generate_single_modulation_iq(mod, 512, 2_400_000, snr_db=30, seed=42)
    t = np.arange(len(i_s)) / 2_400_000 * 1e6   # µs
    ax = axes[idx]
    ax.plot(t[:200], i_s[:200], color='#06b6d4', linewidth=0.8, label='I')
    ax.plot(t[:200], q_s[:200], color='#a855f7', linewidth=0.8, label='Q', alpha=0.7)
    ax.set_title(mod, fontsize=10, fontweight='bold')
    ax.set_xlabel('µs', fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=7)
    ax.set_facecolor('#0f172a')

fig.patch.set_facecolor('#020617')
plt.suptitle('IQ Waveforms by Modulation Type', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('../datasets/waveforms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Build dataset and train ────────────────────────────────────────────────
from scripts.train_model import build_dataset

X, y = build_dataset()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred, labels=sorted(DEMO_MODULATIONS))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(DEMO_MODULATIONS),
            yticklabels=sorted(DEMO_MODULATIONS), ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Modulation Classifier')
plt.tight_layout()
plt.savefig('../datasets/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────
importances = clf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(importances)), importances[sorted_idx], color='#06b6d4', alpha=0.85)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([FEATURE_NAMES[i] for i in sorted_idx], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Importance')
ax.set_title('Random Forest Feature Importances')
plt.tight_layout()
plt.savefig('../datasets/feature_importance.png', dpi=150)
plt.show()